# V3 Pareto Analysis: 2Q Gate Reduction vs Depth Reduction

This notebook analyzes the reward-weight sweep designed for the 2Q/depth tradeoff.

What it does:
- loads `paredo_2q_depth_search.csv` by default
- filters to successful PPO runs from the Pareto sweep
- averages repeated seeds for each `(two_qubit_weight, depth_weight)` pair
- finds the validation Pareto frontier for maximizing 2Q reduction and depth reduction
- reports the same validation-selected frontier models on the test split
- makes paper-friendly Pareto, generalization, and weight-grid plots


## 1. Config

Edit these constants first. The default filename keeps the spelling you used when starting the sweep; if you rename the file later, change `RESULTS_FILE_NAME`.

In [ ]:
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="ticks", context="paper")
plt.rcParams["figure.dpi"] = 130

# Input/output files.
RESULTS_FILE_NAME = "paredo_2q_depth_search.csv"
FIGURE_DIR_NAME = "pareto_2q_depth_analysis_figures"
SAVE_FIGURES = True

# Pareto selection split. Keep this as validation for honest model selection.
PARETO_SPLIT = "validation"
# Reporting split used by some tables/plots. You can set this to validation or test.
REPORT_SPLIT = "test"

# Optional run filters. Set a value to None if you do not want to filter on it.
FILTERS = {
    "algorithm_name": "ppo",
    "feature_batch": "plus_all_features",
    "reward_type": "mixed_weighted",
    "runtime_weight": 0.0,
    "final_reward_weight": 1.0,
}

# Figure colors. Set both background colors to '#FFFFFF' for plain white figures.
FIGURE_BACKGROUND_COLOR = "#FFFFFF"
AXES_BACKGROUND_COLOR = "#F7F9F8"
TEXT_COLOR = "#24333D"
BORDER_COLOR = "#657681"
GRID_COLOR = "#D4DCE0"
PARETO_LINE_COLOR = "#202A33"
PARETO_POINT_COLOR = "#D95F02"
ALL_POINT_COLOR = "#4F7F8C"

# Font sizes used across all plots in this notebook.
TITLE_SIZE = 15
SUBTITLE_SIZE = 12
AXIS_LABEL_SIZE = 11
TICK_LABEL_SIZE = 9
LEGEND_SIZE = 8
ANNOTATION_SIZE = 7
HEATMAP_CELL_SIZE = 7


def resolve_notebook_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "machine_learning" / "RL" / "two_opt_chain_rl_v4",
    ]
    for candidate in candidates:
        if candidate.is_dir() and (candidate / RESULTS_FILE_NAME).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {RESULTS_FILE_NAME}. Run this notebook from the repo root "
        "or from machine_learning/RL/two_opt_chain_rl_v4."
    )


NOTEBOOK_DIR = resolve_notebook_dir()
RESULTS_CSV = NOTEBOOK_DIR / RESULTS_FILE_NAME
FIGURE_DIR = NOTEBOOK_DIR / FIGURE_DIR_NAME
FIGURE_DIR.mkdir(exist_ok=True)


def style_axes(ax, grid_axis="both"):
    ax.figure.set_facecolor(FIGURE_BACKGROUND_COLOR)
    ax.set_facecolor(AXES_BACKGROUND_COLOR)
    ax.set_axisbelow(True)
    ax.tick_params(axis="both", colors=TEXT_COLOR, labelsize=TICK_LABEL_SIZE, width=0.8)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(BORDER_COLOR)
        spine.set_linewidth(0.8)
    ax.grid(False)
    if grid_axis in {"x", "both"}:
        ax.xaxis.grid(True, color=GRID_COLOR, linewidth=0.65, alpha=0.75)
    if grid_axis in {"y", "both"}:
        ax.yaxis.grid(True, color=GRID_COLOR, linewidth=0.65, alpha=0.75)


def save_figure(fig, stem):
    if not SAVE_FIGURES:
        return
    fig.savefig(FIGURE_DIR / f"{stem}.png", bbox_inches="tight", dpi=300, facecolor=fig.get_facecolor())
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", bbox_inches="tight", facecolor=fig.get_facecolor())


print(f"Loading: {RESULTS_CSV}")
print(f"Saving figures to: {FIGURE_DIR}")


## 2. Load And Clean Runs

This keeps the successful runs that match the sweep settings, converts metric columns to numbers, and keeps failed rows visible for debugging.

In [ ]:
raw_results = pd.read_csv(RESULTS_CSV)
raw_results = raw_results.replace({"": np.nan})

numeric_columns = [
    "algorithm_seed",
    "runtime_weight",
    "two_qubit_weight",
    "depth_weight",
    "final_reward_weight",
    "total_timesteps",
    "grid_search_timesteps",
    "grid_search_candidates",
    "elapsed_seconds",
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
]
for column in numeric_columns:
    if column in raw_results.columns:
        raw_results[column] = pd.to_numeric(raw_results[column], errors="coerce")

failed_runs = raw_results[raw_results["status"].astype(str).str.lower() != "success"].copy()
successful_runs = raw_results[raw_results["status"].astype(str).str.lower() == "success"].copy()

for column, expected in FILTERS.items():
    if expected is None or column not in successful_runs.columns:
        continue
    if isinstance(expected, float):
        successful_runs = successful_runs[np.isclose(successful_runs[column].astype(float), expected)]
    else:
        successful_runs = successful_runs[successful_runs[column].astype(str) == str(expected)]

print(f"Raw rows: {len(raw_results)}")
print(f"Successful rows after filters: {len(successful_runs)}")
print(f"Failed/non-success rows: {len(failed_runs)}")

if not failed_runs.empty:
    display(failed_runs[[
        "run_index",
        "algorithm_seed",
        "two_qubit_weight",
        "depth_weight",
        "error_message",
    ]].head(20))

if successful_runs.empty:
    raise ValueError(
        "No successful Pareto-sweep rows found yet. This is expected if only the invalid "
        "2Q=0/depth=0 control row has run so far. Re-run this cell after the sweep finishes."
    )

display(successful_runs.head())


## 3. Build Seed-Level And Weight-Averaged Tables

`seed_level` keeps each seed separate. `comparison` averages the three seeds for each reward-weight pair, which is usually the cleanest table for plots.

In [ ]:
def make_weight_label(row):
    return f"2Q={row['two_qubit_weight']:g}, Depth={row['depth_weight']:g}"


def weight_ratio(row):
    total = float(row["two_qubit_weight"]) + float(row["depth_weight"])
    if total == 0:
        return np.nan
    return float(row["depth_weight"]) / total


seed_group_columns = [
    "algorithm_name",
    "feature_batch",
    "reward_type",
    "runtime_weight",
    "two_qubit_weight",
    "depth_weight",
    "final_reward_weight",
    "algorithm_seed",
]

metric_columns = [
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
]

seed_level = (
    successful_runs
    .groupby(seed_group_columns, as_index=False)
    .agg(
        **{column: (column, "mean") for column in metric_columns},
        model_path=("model_path", "last"),
        run_count=("status", "size"),
    )
)

weight_group_columns = [column for column in seed_group_columns if column != "algorithm_seed"]
comparison = (
    seed_level
    .groupby(weight_group_columns, as_index=False)
    .agg(
        **{column: (column, "mean") for column in metric_columns},
        tested_seed_count=("algorithm_seed", "nunique"),
        tested_seeds=("algorithm_seed", lambda values: ", ".join(str(int(value)) for value in sorted(values.dropna().unique()))),
        model_paths=("model_path", lambda values: " | ".join(str(value) for value in values.dropna())),
    )
)

for frame in [seed_level, comparison]:
    frame["weight_label"] = frame.apply(make_weight_label, axis=1)
    frame["depth_weight_share"] = frame.apply(weight_ratio, axis=1)
    frame["validation_2q"] = frame["validation_avg_two_qubit_gate_reduction_pct"]
    frame["validation_depth"] = frame["validation_avg_depth_reduction_pct"]
    frame["test_2q"] = frame["test_avg_two_qubit_gate_reduction_pct"]
    frame["test_depth"] = frame["test_avg_depth_reduction_pct"]
    frame["validation_runtime"] = frame["validation_avg_runtime_seconds"]
    frame["test_runtime"] = frame["test_avg_runtime_seconds"]

summary_columns = [
    "weight_label",
    "tested_seed_count",
    "tested_seeds",
    "validation_2q",
    "validation_depth",
    "test_2q",
    "test_depth",
    "validation_runtime",
    "test_runtime",
]

display(
    comparison
    .sort_values(["two_qubit_weight", "depth_weight"])
    .reset_index(drop=True)[summary_columns]
)


## 4. Pareto Frontier

A row is Pareto-optimal if no other row has both greater-or-equal 2Q reduction and greater-or-equal depth reduction, with at least one metric strictly better.

In [ ]:
def mark_pareto_front(df, x_col, y_col):
    values = df[[x_col, y_col]].to_numpy(dtype=float)
    is_pareto = np.ones(len(values), dtype=bool)
    for index, point in enumerate(values):
        if np.isnan(point).any():
            is_pareto[index] = False
            continue
        dominates = (
            (values[:, 0] >= point[0])
            & (values[:, 1] >= point[1])
            & ((values[:, 0] > point[0]) | (values[:, 1] > point[1]))
        )
        is_pareto[index] = not np.nan_to_num(dominates, nan=False).any()
    return is_pareto


pareto_x_col = f"{PARETO_SPLIT}_2q"
pareto_y_col = f"{PARETO_SPLIT}_depth"
comparison["is_validation_pareto"] = mark_pareto_front(comparison, pareto_x_col, pareto_y_col)

pareto_front = (
    comparison[comparison["is_validation_pareto"]]
    .sort_values([pareto_x_col, pareto_y_col])
    .reset_index(drop=True)
)

pareto_table_columns = [
    "weight_label",
    "tested_seed_count",
    "tested_seeds",
    "validation_2q",
    "validation_depth",
    "test_2q",
    "test_depth",
    "validation_runtime",
    "test_runtime",
]

print(f"Validation Pareto-front points: {len(pareto_front)}")
display(pareto_front[pareto_table_columns])


## 5. Best Models And Knee Point

The knee point is the validation Pareto model closest to the ideal validation point: max 2Q and max depth.

In [ ]:
def normalized_distance_to_ideal(df, x_col, y_col):
    x = pd.to_numeric(df[x_col], errors="coerce")
    y = pd.to_numeric(df[y_col], errors="coerce")
    x_range = x.max() - x.min()
    y_range = y.max() - y.min()
    if x_range == 0:
        x_range = 1.0
    if y_range == 0:
        y_range = 1.0
    return np.sqrt(((x.max() - x) / x_range) ** 2 + ((y.max() - y) / y_range) ** 2)


comparison["validation_balanced_score"] = (
    comparison["validation_2q"].rank(pct=True)
    + comparison["validation_depth"].rank(pct=True)
) / 2

pareto_front = pareto_front.copy()
pareto_front["distance_to_validation_ideal"] = normalized_distance_to_ideal(
    pareto_front,
    "validation_2q",
    "validation_depth",
)

winner_rows = []
winner_specs = [
    ("Best validation 2Q", comparison, "validation_2q", "idxmax"),
    ("Best validation depth", comparison, "validation_depth", "idxmax"),
    ("Best balanced validation rank", comparison, "validation_balanced_score", "idxmax"),
    ("Validation Pareto knee", pareto_front, "distance_to_validation_ideal", "idxmin"),
    ("Best test 2Q among validation Pareto", pareto_front, "test_2q", "idxmax"),
    ("Best test depth among validation Pareto", pareto_front, "test_depth", "idxmax"),
]

for category, frame, column, selector in winner_specs:
    if frame.empty or frame[column].isna().all():
        continue
    row = frame.loc[getattr(frame[column], selector)()]
    winner_rows.append({
        "category": category,
        "weight_label": row["weight_label"],
        "validation_2q": row["validation_2q"],
        "validation_depth": row["validation_depth"],
        "test_2q": row["test_2q"],
        "test_depth": row["test_depth"],
        "validation_runtime": row["validation_runtime"],
        "test_runtime": row["test_runtime"],
    })

winners = pd.DataFrame(winner_rows)
display(winners)


## 6. Main Pareto Scatter Plot

This is the main figure: all averaged reward-weight settings are shown, and validation Pareto points are outlined and connected.

In [ ]:
def plot_pareto_scatter(df, front, split, filename):
    x_col = f"{split}_2q"
    y_col = f"{split}_depth"
    fig, ax = plt.subplots(figsize=(7.2, 5.4), facecolor=FIGURE_BACKGROUND_COLOR)
    style_axes(ax, grid_axis="both")

    scatter = ax.scatter(
        df[x_col],
        df[y_col],
        c=df["depth_weight_share"],
        cmap="viridis",
        s=70,
        edgecolor=BORDER_COLOR,
        linewidth=0.55,
        alpha=0.82,
        label="Reward-weight setting",
    )

    if not front.empty:
        ordered_front = front.sort_values(x_col)
        ax.plot(
            ordered_front[x_col],
            ordered_front[y_col],
            color=PARETO_LINE_COLOR,
            linewidth=1.25,
            alpha=0.85,
            label="Validation Pareto front",
        )
        ax.scatter(
            ordered_front[x_col],
            ordered_front[y_col],
            s=115,
            facecolor="none",
            edgecolor=PARETO_POINT_COLOR,
            linewidth=1.4,
            label="Pareto-optimal",
        )
        for _, row in ordered_front.iterrows():
            ax.annotate(
                row["weight_label"],
                (row[x_col], row[y_col]),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=ANNOTATION_SIZE,
                color=TEXT_COLOR,
            )

    ax.set_title(f"{split.title()} 2Q-Depth Pareto Analysis", fontsize=TITLE_SIZE, weight="semibold", pad=10)
    ax.set_xlabel(f"{split.title()} 2Q Gate Reduction (%)", fontsize=AXIS_LABEL_SIZE)
    ax.set_ylabel(f"{split.title()} Depth Reduction (%)", fontsize=AXIS_LABEL_SIZE)
    colorbar = fig.colorbar(scatter, ax=ax, pad=0.02)
    colorbar.set_label("Depth-weight share", fontsize=AXIS_LABEL_SIZE, color=TEXT_COLOR)
    colorbar.ax.tick_params(labelsize=TICK_LABEL_SIZE, colors=TEXT_COLOR)
    colorbar.outline.set_edgecolor(BORDER_COLOR)
    ax.legend(loc="best", fontsize=LEGEND_SIZE, frameon=True)
    fig.tight_layout()
    save_figure(fig, filename)
    plt.show()
    plt.close(fig)


plot_pareto_scatter(comparison, pareto_front, PARETO_SPLIT, "validation_pareto_scatter")


## 7. Test Performance Of Validation Pareto Models

This plot uses the models selected by the validation frontier, then places those same models using test metrics.

In [ ]:
plot_pareto_scatter(comparison, pareto_front, REPORT_SPLIT, "test_scatter_validation_pareto_highlighted")


## 8. Validation-To-Test Shift

Each arrow starts at a validation Pareto point and ends at that same reward setting's test result.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5.4), facecolor=FIGURE_BACKGROUND_COLOR)
style_axes(ax, grid_axis="both")

ax.scatter(
    pareto_front["validation_2q"],
    pareto_front["validation_depth"],
    color=ALL_POINT_COLOR,
    s=80,
    edgecolor=BORDER_COLOR,
    linewidth=0.7,
    label="Validation",
)
ax.scatter(
    pareto_front["test_2q"],
    pareto_front["test_depth"],
    color=PARETO_POINT_COLOR,
    s=90,
    edgecolor=BORDER_COLOR,
    linewidth=0.7,
    label="Test",
)

for _, row in pareto_front.iterrows():
    ax.annotate(
        "",
        xy=(row["test_2q"], row["test_depth"]),
        xytext=(row["validation_2q"], row["validation_depth"]),
        arrowprops={"arrowstyle": "->", "color": "#5F6F7A", "lw": 0.9, "alpha": 0.75},
    )
    ax.annotate(
        row["weight_label"],
        (row["test_2q"], row["test_depth"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=ANNOTATION_SIZE,
        color=TEXT_COLOR,
    )

ax.set_title("Validation Pareto Models: Test-Set Shift", fontsize=TITLE_SIZE, weight="semibold", pad=10)
ax.set_xlabel("2Q Gate Reduction (%)", fontsize=AXIS_LABEL_SIZE)
ax.set_ylabel("Depth Reduction (%)", fontsize=AXIS_LABEL_SIZE)
ax.legend(loc="best", fontsize=LEGEND_SIZE, frameon=True)
fig.tight_layout()
save_figure(fig, "validation_to_test_shift")
plt.show()
plt.close(fig)


## 9. Reward-Weight Grid Heatmaps

These make it easier to see whether the reward landscape behaves sensibly as weights change.

In [ ]:
def plot_weight_heatmap(df, value_col, title, filename, cmap="crest", fmt=".1f"):
    pivot = df.pivot_table(
        index="depth_weight",
        columns="two_qubit_weight",
        values=value_col,
        aggfunc="mean",
    ).sort_index(ascending=False)

    fig, ax = plt.subplots(figsize=(7.4, 5.4), facecolor=FIGURE_BACKGROUND_COLOR)
    style_axes(ax, grid_axis=None)
    sns.heatmap(
        pivot,
        annot=True,
        fmt=fmt,
        cmap=cmap,
        linewidths=0.55,
        linecolor=AXES_BACKGROUND_COLOR,
        cbar_kws={"label": title},
        annot_kws={"size": HEATMAP_CELL_SIZE},
        ax=ax,
    )
    ax.set_title(title, fontsize=TITLE_SIZE, weight="semibold", pad=10)
    ax.set_xlabel("2Q reward weight", fontsize=AXIS_LABEL_SIZE)
    ax.set_ylabel("Depth reward weight", fontsize=AXIS_LABEL_SIZE)
    ax.tick_params(axis="x", labelsize=TICK_LABEL_SIZE, rotation=0)
    ax.tick_params(axis="y", labelsize=TICK_LABEL_SIZE, rotation=0)
    colorbar = ax.collections[0].colorbar
    colorbar.ax.tick_params(labelsize=TICK_LABEL_SIZE, colors=TEXT_COLOR)
    colorbar.set_label(title, fontsize=AXIS_LABEL_SIZE, color=TEXT_COLOR)
    colorbar.outline.set_edgecolor(BORDER_COLOR)
    fig.tight_layout()
    save_figure(fig, filename)
    plt.show()
    plt.close(fig)


plot_weight_heatmap(
    comparison,
    "validation_2q",
    "Validation 2Q Gate Reduction (%)",
    "validation_2q_weight_heatmap",
    cmap="crest",
)
plot_weight_heatmap(
    comparison,
    "validation_depth",
    "Validation Depth Reduction (%)",
    "validation_depth_weight_heatmap",
    cmap="mako",
)

pareto_membership = comparison.copy()
pareto_membership["pareto_member"] = pareto_membership["is_validation_pareto"].astype(int)
plot_weight_heatmap(
    pareto_membership,
    "pareto_member",
    "Validation Pareto Membership",
    "validation_pareto_membership_heatmap",
    cmap=sns.light_palette(PARETO_POINT_COLOR, as_cmap=True),
    fmt=".0f",
)


## 10. Seed Stability

This plot shows whether the tradeoff pattern is stable across seeds. It only becomes especially meaningful after all three seeds finish.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5.4), facecolor=FIGURE_BACKGROUND_COLOR)
style_axes(ax, grid_axis="both")

sns.scatterplot(
    data=seed_level,
    x="validation_2q",
    y="validation_depth",
    hue="depth_weight_share",
    palette="viridis",
    style="algorithm_seed",
    s=70,
    edgecolor=BORDER_COLOR,
    linewidth=0.55,
    alpha=0.75,
    ax=ax,
)
ax.scatter(
    comparison["validation_2q"],
    comparison["validation_depth"],
    s=95,
    facecolor="none",
    edgecolor=PARETO_LINE_COLOR,
    linewidth=1.0,
    label="Seed mean",
)
ax.set_title("Seed-Level Validation Tradeoffs", fontsize=TITLE_SIZE, weight="semibold", pad=10)
ax.set_xlabel("Validation 2Q Gate Reduction (%)", fontsize=AXIS_LABEL_SIZE)
ax.set_ylabel("Validation Depth Reduction (%)", fontsize=AXIS_LABEL_SIZE)
ax.legend(loc="best", fontsize=LEGEND_SIZE, frameon=True)
fig.tight_layout()
save_figure(fig, "seed_level_validation_tradeoffs")
plt.show()
plt.close(fig)


## 11. Paper Table Export

This creates a compact table for the validation Pareto front and prints a LaTeX version.

In [ ]:
paper_table = pareto_front[[
    "weight_label",
    "tested_seed_count",
    "validation_2q",
    "validation_depth",
    "test_2q",
    "test_depth",
    "test_runtime",
]].copy()

paper_table = paper_table.rename(columns={
    "weight_label": "Reward Weights",
    "tested_seed_count": "Seeds",
    "validation_2q": "Val. 2Q (%)",
    "validation_depth": "Val. Depth (%)",
    "test_2q": "Test 2Q (%)",
    "test_depth": "Test Depth (%)",
    "test_runtime": "Test Runtime (s)",
})

for column in ["Val. 2Q (%)", "Val. Depth (%)", "Test 2Q (%)", "Test Depth (%)", "Test Runtime (s)"]:
    paper_table[column] = pd.to_numeric(paper_table[column], errors="coerce").map(lambda value: f"{value:.2f}")

display(paper_table)

latex_table = paper_table.to_latex(
    index=False,
    escape=True,
    caption="Validation Pareto-front reward settings and held-out test performance.",
    label="tab:pareto_2q_depth",
    column_format="lrrrrrr",
)
print(latex_table)

if SAVE_FIGURES:
    print(f"Saved figures to: {FIGURE_DIR}")
